# WW Prediction – Hybrid v3 LOEO
Pipeline modulare identica al monolite `WW_hybrid_v3_loeo.py`.

**Struttura:**
```
STEP 1  → compute_sensor_residuals
STEP 2  → create_base_features
STEP 3  → aggregate_by_cycle
STEP 4a → estimate_ww_period_per_engine  [ORIGINALE]
STEP 4b → add_hpc_ww_recovery_feature    [MathWorks]
STEP 4c → add_residual_shock_features    [ORIGINALE]
STEP 4d → add_periodic_and_residual_features
STEP 5  → select_features  (dentro LOEO, su train)
STEP 6  → run_loeo
STEP 7  → plot_results + save_results
```

In [37]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
# Assicurati che src/ sia nel path
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd

from src.utils import load_config

# Feature engineering
from src.phm_rul.features.residuals    import compute_sensor_residuals
from src.phm_rul.features.base         import create_base_features
from src.phm_rul.features.aggregate    import aggregate_by_cycle
from src.phm_rul.features.periodic_fft import estimate_ww_period_per_engine
from src.phm_rul.features.recovery     import add_hpc_ww_recovery_feature
from src.phm_rul.features.shock        import add_residual_shock_features
from src.phm_rul.features.rolling      import add_periodic_and_residual_features

# Pipeline LOEO + reporting
from src.phm_rul.pipeline   import run_loeo
from src.phm_rul.reporting  import plot_results, save_results

In [38]:
# ── Config ──────────────────────────────────────────────────────────────
cfg = load_config('configs/config.yaml')

WINDOW_SIZE = 30
TOP_K       = 70
OUT_DIR     = f'artifacts/hybrid_v3_loeo_{WINDOW_SIZE}'


## STEP 1–4: Feature Pipeline (snapshot-level)

In [39]:
df = pd.read_csv(cfg['data']['train_clean_csv'])
print(f'Loaded: {df.shape}, ESNs: {sorted(df["ESN"].unique())}')

total_nan = df.isna().sum().sum()
if total_nan:
    print(f'  ⚠️  RAW: {total_nan} NaN found:')
    print(df.isna().sum()[df.isna().sum() > 0].to_string())

Loaded: (59702, 28), ESNs: [np.int64(101), np.int64(102), np.int64(103), np.int64(104)]
  ⚠️  RAW: 4577 NaN found:
Sensed_WFuel                                       697
Sensed_Core_Speed                                  697
Sensed_T3                                           60
Sensed_Ps3                                          60
Sensed_T45                                         749
Sensed_T5                                          751
HPC_Eff_Index_clean                                 51
ratio_T3_T45                                       756
tri_ratio_diff_Sensed_Mach_Sensed_T3_Sensed_T45    756


In [40]:
print('\n' + '='*70)
print('STEP 1: SENSOR RESIDUALS')
print('='*70)
df, res_cols = compute_sensor_residuals(df)


STEP 1: SENSOR RESIDUALS

STEP 1: SENSOR RESIDUALS (per-engine, Ridge regression)
  ESN 101: 14738 valid snapshots
  ESN 102: 15044 valid snapshots
  ESN 103: 14856 valid snapshots
  ESN 104: 15064 valid snapshots
  Residual cols: ['Sensed_T3_res', 'Sensed_T45_res', 'Sensed_Ps3_res', 'Sensed_WFuel_res', 'Sensed_Core_Speed_res', 'Sensed_T25_res']


In [41]:
print('\n' + '='*70)
print('STEP 2: BASE FEATURES')
print('='*70)
df = create_base_features(df)


STEP 2: BASE FEATURES

  Creating base features...
    Created 13 base features


In [42]:
print('\n' + '='*70)
print('STEP 3: AGGREGATING BY CYCLE')
print('='*70)
df_agg = aggregate_by_cycle(df, res_cols)


STEP 3: AGGREGATING BY CYCLE

  Aggregating snapshots → cycles...
    59,702 snapshots → 8,004 cycles


In [43]:
# STEP 4: tutte le feature originali
df_agg = estimate_ww_period_per_engine(df_agg)          # 4a FFT [ORIGINALE]
df_agg = add_hpc_ww_recovery_feature(df_agg)            # 4b MathWorks
df_agg = add_residual_shock_features(df_agg)            # 4c Shock [ORIGINALE]
df_agg = add_periodic_and_residual_features(df_agg, res_cols)  # 4d Rolling

print(f'\ndf_agg shape: {df_agg.shape}')


STEP 4a: ADAPTIVE WW PERIOD DETECTION (FFT) [ORIGINALE]
  ESN 101: FFT-estimated WW period = 1000 cycles
  ESN 102: FFT-estimated WW period = 1000 cycles
  ESN 103: FFT-estimated WW period = 600 cycles
  ESN 104: FFT-estimated WW period = 1000 cycles

STEP 4b: HPC-WW RECOVERY FEATURE (MathWorks)
  ESN 101: 4 WW recovery events detected
  ESN 102: 29 WW recovery events detected
  ESN 103: 10 WW recovery events detected
  ESN 104: 44 WW recovery events detected

STEP 4c: RESIDUAL SHOCK DETECTOR [ORIGINALE]
  ESN 101: 10 shocks detected, avg interval=121 cycles
  ESN 102: 21 shocks detected, avg interval=57 cycles
  ESN 103: 16 shocks detected, avg interval=126 cycles
  ESN 104: 34 shocks detected, avg interval=60 cycles

STEP 4d: PERIODIC + RESIDUAL FEATURES
    ESN 101: periodic + residual features added
    ESN 102: periodic + residual features added
    ESN 103: periodic + residual features added
    ESN 104: periodic + residual features added

  Dataset shape: (8004, 254)

df_agg sh

## STEP 5–6: LOEO
> `select_features` viene chiamata **dentro** `run_loeo` ad ogni fold, solo su `df_train`. Questo replica esattamente il comportamento del monolite e previene il leakage.

In [44]:
fold_results = run_loeo(
    df_agg,
    res_cols    = res_cols,
    window_size = WINDOW_SIZE,
    top_k       = TOP_K,
)


STEP 6: LOEO | window=30

──────────────────────────────────────────────────────────────────────
FOLD: Leave out ESN 101
──────────────────────────────────────────────────────────────────────

STEP 5: FEATURE SELECTION (RF importance, top_k=70)
  Candidates: 251 | Selected: 70
  Top 10: ['Sensed_T3_res_slope50', 'Sensed_T3_res_slope20', 'ww_cos_1000', 'Sensed_T3_res_slope100', 'ww_sin_1000', 'Sensed_T3_res_rstd_50', 'Sensed_T45_res_rstd_50', 'shock_magnitude_cumsum', 'Sensed_T25_res_rstd_100', 'hpc_hi_slope10']
  Force-added: Sensed_T45_res
  Force-added: T45res_x_tempgrad
  Force-added: T45res_x_heff
  Force-added: relative_cycle
  Force-added: cycles_since_start
  Force-added: Sensed_Ps3_res_cumsum_norm
  Force-added: Sensed_Core_Speed_res_cumsum_norm
  Force-added: temp_gradient_cumsum_norm
  Force-added: hpc_hi_recovery
  Force-added: ww_period_est
  Using 80 features for LOEO
  Train: (5913, 480), y=[0,1160]
  Test:  (1971, 480),  y=[0,1150]

  Training Full Ensemble (4 models + 

## STEP 7: Visualizzazione e salvataggio

In [45]:
plot_results(fold_results, OUT_DIR)
df_res = save_results(fold_results, OUT_DIR)
print(f'\n📁 Results saved → {OUT_DIR}/')


  ✓ Plots saved → artifacts/hybrid_v3_loeo_30/

FINAL RESULTS
 left_out_esn        mae         twe       r2  baseline_mae  improvement
          101 177.552840 1355.433182 0.325749    252.359389    29.642863
          102 166.972863 2742.793121 0.469203    250.761872    33.413775
          103 229.163683 1015.943188 0.068776    250.342184     8.459821
          104 218.483693 1451.168285 0.118127    256.120189    14.694857

  Avg MAE : 198.0
  Avg TWE : 1641.3344
  Avg R²  : 0.245
  Avg Impr: +21.6%

📁 Results saved → artifacts/hybrid_v3_loeo_30/


In [15]:
# Riepilogo rapido
print('\n--- SUMMARY ---')
print(df_res[['left_out_esn','mae','twe','r2','improvement']].to_string(index=False))


--- SUMMARY ---
 left_out_esn        mae         twe       r2  improvement
          101 169.186236 1533.196057 0.387110    33.058394
          102 172.562660 2238.287847 0.420270    31.169470
          103 227.984594  889.249975 0.004633     8.965066
          104 214.757677 1744.570202 0.139331    16.288113
